In [1]:
import pandas as pd

df = pd.read_csv("../data/fake_job_postings.csv")

text_cols = ["title", "company_profile", "description", "requirements", "benefits"]

print("Missing BEFORE filling:")
print(df[text_cols].isnull().sum())

df[text_cols] = df[text_cols].fillna("")

print("\nMissing AFTER filling:")
print(df[text_cols].isnull().sum())

Missing BEFORE filling:
title                 0
company_profile    3308
description           1
requirements       2696
benefits           7212
dtype: int64

Missing AFTER filling:
title              0
company_profile    0
description        0
requirements       0
benefits           0
dtype: int64


In [3]:
import html

df["combined_text"] = (
    df["title"] + " " +
    df["company_profile"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["benefits"]
)

df["combined_text"] = df["combined_text"].apply(html.unescape)
df["combined_text"] = df["combined_text"].str.replace(r"\s+", " ", regex=True).str.strip()

print("Sample combined text (first 400 characters):\n")
print(df["combined_text"].iloc[0][:400])
print("\nAverage length:", int(df["combined_text"].str.len().mean()), "characters")

Sample combined text (first 400 characters):

Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate home cooks, and give them everything they need in one place.We have a top editorial, business, and engineering team. We're focused on using technology to find new and better ways to connect people around their specific food interests, and to offer them superb, highly

Average length: 2647 characters


In [4]:
rows_before = len(df)

df = df[df["combined_text"].str.len() > 0]

rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Empty rows removed:", rows_before - rows_after)

Rows before: 17880
Rows after:  17880
Empty rows removed: 0


In [5]:
rows_before = len(df)

df = df.drop_duplicates(subset="combined_text")

rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Duplicates removed:", rows_before - rows_after)

print("\nReal vs fake after removing duplicates:")
print(df["fraudulent"].value_counts())

Rows before: 17880
Rows after:  15879
Duplicates removed: 2001

Real vs fake after removing duplicates:
fraudulent
0    15166
1      713
Name: count, dtype: int64


In [6]:
df = df[["combined_text", "fraudulent"]].reset_index(drop=True)

print("Final shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Final shape: (15879, 2)
Columns: ['combined_text', 'fraudulent']


,combined_text,fraudulent
0,"Marketing Intern We're Food52, and we've creat...",0
1,Customer Service - Cloud Video Production 90 S...,0
2,Commissioning Machinery Assistant (CMA) Valor ...,0
3,Account Executive - Washington DC Our passion ...,0
4,Bill Review Manager SpotSource Solutions LLC i...,0


In [7]:
df.to_csv("../data/cleaned_job_postings.csv", index=False)

print("Saved to data/cleaned_job_postings.csv")
print("Rows saved:", len(df))

Saved to data/cleaned_job_postings.csv
Rows saved: 15879
